# Blackboard benchmark pilot — saved results in Google Drive

Run actual persona agents with one local Qwen model on three selected official KramaBench CSV tasks. Start with **one task**, then try the three-task batch and reconnect/resume check.

**Status:** local offline/official-asset checks pass. Fresh managed Colab GPU execution and Drive reconnection acceptance are pending. Recovery is disabled. MedAgentBench is unavailable until its official local server/evaluator are verified. Scores cover this small profile, not an official full benchmark aggregate.

Choose **Runtime → Change runtime type → GPU**, then run cells from top to bottom. No paid API key or external server is needed. See the [usage guide](https://github.com/nikhil0744/intelligible-blackboard-architecture/blob/feature/colab-benchmarks/docs/guides/colab-benchmarks.md).

In [ ]:
#@title 1. Install the pinned project revision
import importlib.metadata as metadata
import os
import re
import subprocess
import sys
from pathlib import Path

PROJECT_REVISION = "75926469f49f3c987b8a82b7e4f6c75f64c7dd5a" #@param {type:"string"}
PROJECT_DIR = Path("/content/intelligible-blackboard")
REPO_URL = "https://github.com/nikhil0744/intelligible-blackboard-architecture.git"
if not re.fullmatch(r"[0-9a-f]{40}", PROJECT_REVISION):
    raise ValueError("Use an immutable 40-character project commit, not a branch name.")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(PROJECT_DIR)], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "--detach", PROJECT_REVISION], check=True)
actual_revision = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_revision != PROJECT_REVISION:
    raise ValueError("This runtime contains another project revision. Start a fresh runtime; do not replace code during a saved experiment.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR) + "[colab]"], check=True)
os.chdir(PROJECT_DIR)
for name in ("numpy", "pandas", "pydantic"):
    loaded = sys.modules.get(name)
    if loaded is not None and getattr(loaded, "__version__", None) != metadata.version(name):
        raise RuntimeError("A loaded package changed during installation. Restart the Colab session, then rerun the notebook.")
print("Project revision:", actual_revision)
print("Python:", sys.version.split()[0])

## Choose where to save and what to run

Drive is required for normal persistent runs. Temporary smoke mode explicitly uses disposable runtime storage. A **new run** creates a separate experiment each time; **resume** continues the exact saved selection/settings and skips committed results, including incorrect answers. Copy the printed run folder into `RESUME_RUN_DIR` after reconnecting.

For a resume check, set `STOP_AFTER_SAVED = 1`, run three tasks, reconnect in a new runtime, and resume that folder with `STOP_AFTER_SAVED = 0`. Keep this project's pinned revision and original CPU/GPU mode. The saved manifest lists required package versions; incompatible runtimes are rejected.

In [ ]:
#@title 2. Experiment controls and Drive
from google.colab import drive
from benchmarks.batch import validate_destination

TEMPORARY_SMOKE = False #@param {type:"boolean"}
CPU_SMOKE = False #@param {type:"boolean"}
SAVE_ROOT = "/content/drive/MyDrive/blackboard_benchmarks" #@param {type:"string"}
BENCHMARK = "kramabench" #@param ["kramabench", "medagentbench"]
TASK_COUNT = 1 #@param [1, 2, 3] {type:"raw"}
SEED = 0 #@param {type:"integer"}
MODE = "new run" #@param ["new run", "resume"]
RESUME_RUN_DIR = "" #@param {type:"string"}
STOP_AFTER_SAVED = 0 #@param {type:"integer"}
RETRY_FAILED_ATTEMPTS = False #@param {type:"boolean"}

if TEMPORARY_SMOKE:
    ROOT = Path("/content/temporary-blackboard-benchmarks")
    print("Temporary smoke mode: download results before disconnecting; runtime storage can disappear.")
else:
    drive.mount("/content/drive")
    ROOT = Path(SAVE_ROOT)
validate_destination(ROOT, temporary=TEMPORARY_SMOKE)
CACHE_DIR, RESULTS_DIR = ROOT / "cache", ROOT / "runs"
if MODE == "resume" and not RESUME_RUN_DIR.strip():
    raise ValueError("Paste the saved run folder into RESUME_RUN_DIR.")
if STOP_AFTER_SAVED < 0:
    raise ValueError("STOP_AFTER_SAVED must be zero or a positive count.")
if MODE == "new run" and RETRY_FAILED_ATTEMPTS:
    raise ValueError("Retry failures applies only to resume mode.")
print("Save root:", ROOT)
print("Selection:", TASK_COUNT, "task(s) for a new run; resume uses its saved selection.")

In [ ]:
#@title 3. Prepare verified data, local Ollama and the pinned Qwen model
import json
from benchmarks.preparation import resources, prepare, PreparedBenchmark
from benchmarks.colab_setup import ensure_service
from benchmarks.batch import check_readiness, source_identity, package_identity
from benchmarks.persistence import RunStore
from llm_broker.config import InferenceSettings, build_broker

specifications = resources()
if not specifications[BENCHMARK].get("available", True):
    raise RuntimeError(specifications[BENCHMARK]["reason"])
if MODE == "resume":
    previous_store = RunStore(Path(RESUME_RUN_DIR))
    saved = previous_store.manifest["configuration"]
    if saved["code_sha256"] != source_identity()["source_sha256"]:
        raise ValueError("Resume needs the original code revision from manifest.json; start a compatible runtime first.")
    if saved["packages"] != package_identity():
        raise ValueError("Resume needs the Python/package versions in manifest.json. Restore those versions or create a new run.")
    if saved["cpu_smoke"] != CPU_SMOKE:
        raise ValueError("Select the same CPU_SMOKE setting as the saved run.")
    settings = InferenceSettings(**saved["inference"])
else:
    settings = InferenceSettings(model=specifications["runtime"]["model"], ollama_host="http://127.0.0.1:11434")
prepare(CACHE_DIR, BENCHMARK)
prepared = PreparedBenchmark.open(CACHE_DIR, BENCHMARK)
print(json.dumps(prepared.coverage(), indent=2))
service = ensure_service(cpu_smoke=CPU_SMOKE)
if "broker" in globals():
    broker.close()
broker = build_broker(settings)
readiness = check_readiness(broker, settings, cpu_smoke=CPU_SMOKE)
print("Ready:", settings.model, "digest", readiness["model_digest"])
print("Hardware:", readiness["hardware"])
print("Package versions:", package_identity())
print("Recovery: disabled. MedAgentBench:", specifications["medagentbench"]["reason"])

## Run and inspect the discussion

Progress below shows actual agent contributions and tool activity. Full prompts/responses, repairs and observations are saved in each attempt's event log. A final number is frozen before independent grading; consensus does not mean the answer is correct.

Use the stop button to interrupt an executing task. Resume restarts that task with a new attempt ID and retains its partial records. Stop after an `attempt_saved` message to test resuming completed work. Storage failures stop the batch immediately.

In [ ]:
#@title 4. Execute the shared runner
from benchmarks.batch import run_batch, resume_batch
from benchmarks.runtime import Limits

RUN_DIR = RESUME_RUN_DIR.strip() if MODE == "resume" else ""
saved_in_this_invocation = 0

def show_progress(event):
    global RUN_DIR, saved_in_this_invocation
    kind = event["kind"]
    if kind == "run_created":
        RUN_DIR = event["run_dir"]
        print("SAVE THIS RUN FOLDER:", RUN_DIR, flush=True)
    elif kind == "session_started":
        print("\nTask:", event["task"]["question"], flush=True)
    elif kind == "contribution":
        post = event["contribution"]
        print(f"[{post['agent_id']}] {post['tag']} → {post['payload']['prediction']['claim']}", flush=True)
        print("  Why:", post["payload"]["explanation"]["rationale"][:500], flush=True)
    elif kind == "tool_requested":
        print(f"[{event['agent_id']}] tool {event['name']}: {json.dumps(event['arguments'])[:500]}", flush=True)
    elif kind == "tool_observed":
        print("  Observation:", json.dumps(event["result"], ensure_ascii=False)[:700], flush=True)
    elif kind == "attempt_saved":
        RUN_DIR = event["run_dir"]
        saved_in_this_invocation += 1
        print("SAVED:", event["task_id"], event["status"], "grade:", event["grade"], flush=True)
        if STOP_AFTER_SAVED and saved_in_this_invocation >= STOP_AFTER_SAVED:
            raise KeyboardInterrupt
    elif kind in {"task_skipped", "session_stopped", "run_time_limit", "deadlock_detected"}:
        print(event, flush=True)

try:
    if MODE == "resume":
        store, summary = resume_batch(Path(RUN_DIR), prepared, broker, settings,
            cpu_smoke=CPU_SMOKE, retry_failures=RETRY_FAILED_ATTEMPTS,
            readiness=readiness, observer=show_progress)
    else:
        store, summary = run_batch(prepared, broker, settings, RESULTS_DIR, limit=int(TASK_COUNT),
            seed=SEED, limits=Limits(), cpu_smoke=CPU_SMOKE, temporary=TEMPORARY_SMOKE,
            readiness=readiness, observer=show_progress)
    RUN_DIR = str(store.root)
except KeyboardInterrupt:
    print("Stopped. Saved attempts remain intact. Switch to resume mode to continue this run.")
print("Run folder:", RUN_DIR)

In [ ]:
#@title 5. Rebuild results and read a saved trace
import pandas as pd
from IPython.display import display
from benchmarks.presentation import trace_table, plot_summary

if not RUN_DIR:
    raise ValueError("Run an experiment first, or enter its saved folder in resume mode.")
with RunStore(Path(RUN_DIR)).writer() as stored:
    summary = stored.summarize()
counts = {k: summary[k] for k in ("selected", "started", "completed", "scored", "correct", "incorrect", "ungraded", "timed_out", "errored", "interrupted", "pending")}
print(counts)
print("Success / all selected:", summary["success_rate_all_selected"], "denominator", summary["selected"])
print("Success / scored:", summary["success_rate_scored"], "denominator", summary["scored"])
print("Usage across all attempts (unknown stays null):", summary["usage_all_attempts"])
display(pd.DataFrame([{"task":r["task_id"], "status":r["status"], "consensus":r.get("consensus"),
    "score":r.get("grade",{}).get("score"), "correct":r.get("grade",{}).get("correct"),
    "seconds":r.get("elapsed_s"), "attempt":r.get("attempt_id")} for r in summary["outcomes"]]))
if any(record.get("committed", True) for record in stored.attempts()):
    display(pd.DataFrame(trace_table(RUN_DIR)))
    print("Outcome figure:", plot_summary(RUN_DIR))
else:
    print("No committed attempt yet. Partial event/call logs remain in the run folder.")
print("Files:", RUN_DIR)

In [ ]:
#@title 6. Optional recorded playback and downloadable bundle
import html
import shutil
from google.colab import files
from IPython.display import HTML, display
from benchmarks.presentation import export_viewer

SHOW_PLAYBACK = True #@param {type:"boolean"}
DOWNLOAD_BUNDLE = False #@param {type:"boolean"}
if SHOW_PLAYBACK:
    viewer = export_viewer(Path(RUN_DIR), PROJECT_DIR / "visualizer.html")
    # Run the existing viewer in an iframe; model strings cannot execute in the notebook.
    embedded = html.escape(viewer.read_text(), quote=True)
    display(HTML('<iframe sandbox="allow-scripts" style="width:100%;height:900px;border:0" srcdoc="' + embedded + '"></iframe>'))
    print("Recorded playback:", viewer)
if DOWNLOAD_BUNDLE:
    archive = shutil.make_archive("/content/" + Path(RUN_DIR).name, "zip", RUN_DIR)
    files.download(archive)

## Acceptance and stopping

Keep the run manifest and records as evidence. Verify one GPU task first, then the three-task interruption/resume test in a new runtime. Check that saved grades, task denominators, contributions, tools and targets match the files in Drive. Incorrect answers can still demonstrate valid execution/grading; scripted unit checks cannot establish model performance.

To continue after reconnecting: rerun setup at the same revision, choose **resume**, paste the run folder, set the same CPU/GPU mode and leave `STOP_AFTER_SAVED = 0`. No keep-alive tricks, tunnels or external services are used. MedAgentBench feasibility and the separate recovery milestone remain pending.

In [ ]:
#@title Optional: stop this notebook's Ollama process
# This does not stop a pre-existing service owned by another process.
if "service" in globals():
    service.stop()
if "broker" in globals():
    broker.close()
print("Owned inference service stopped, if one was started. Drive results remain saved.")